# 面试问题：Multi-Agent Debate 怎样避免把相关错误误当成独立共识？

可以直接复述的回答是：第一，多 Agent 数量不等于独立证据数量。第二，记录每个判断的来源和 evidence fingerprint。第三，共享同一监控摘要或同一模型的票应先聚类，再按证据簇计票。第四，平票或证据不足时应 abstain，而不是用更自信的措辞。第五，用历史校准给来源赋权，但不能在当前测试集调权。第六，要展示投票矩阵、相关性、聚类共识和逐事件准确率。下面用五个线上事故分级案例演示。

## 真实案例：支付平台事故严重级别共识

五起脱敏事故需要判断 P1–P4。四个 Agent 中，A 与 B 都读取同一个 metrics dashboard，C 读取客服工单，D 读取部署与审计记录。人工复盘标签用于离线比较；案例不代表真实应急规则，生产分级仍由值班负责人确认。

In [1]:
incidents = [  # 定义五起具有可读症状和人工级别的事故
    {"id": "I1", "summary": "全站支付失败率 82%，持续 12 分钟", "truth": "P1"},  # 全局核心业务中断
    {"id": "I2", "summary": "API 延迟升高但成功率正常，影响约 18% 请求", "truth": "P2"},  # 部分用户显著退化
    {"id": "I3", "summary": "单一企业租户无法下载账单", "truth": "P3"},  # 单租户功能故障
    {"id": "I4", "summary": "计划维护窗口内监控告警，无用户影响", "truth": "P4"},  # 无实际影响的计划事件
    {"id": "I5", "summary": "数据库误删导致订单状态不可恢复", "truth": "P1"},  # 数据完整性严重事故
]  # 结束五起事故评测
agents = [  # 定义四个 Agent 的证据渠道
    {"id": "A", "source": "metrics", "description": "主监控 Dashboard"},  # 读取实时指标
    {"id": "B", "source": "metrics", "description": "同一 Dashboard 的摘要副本"},  # 与 A 完全相关而非独立观察
    {"id": "C", "source": "support", "description": "客服影响工单"},  # 独立用户影响证据
    {"id": "D", "source": "audit", "description": "部署与数据审计"},  # 独立变更和完整性证据
]  # 结束四个 Agent 定义
votes = {  # 定义每起事故的四个离线判断
    "I1": {"A": "P1", "B": "P1", "C": "P1", "D": "P1"},  # 所有证据一致
    "I2": {"A": "P1", "B": "P1", "C": "P2", "D": "P2"},  # 共享监控摘要过度升级
    "I3": {"A": "P2", "B": "P2", "C": "P3", "D": "P3"},  # 共享监控忽略单租户范围
    "I4": {"A": "P2", "B": "P2", "C": "P4", "D": "P4"},  # 共享监控忽略维护窗口
    "I5": {"A": "P1", "B": "P1", "C": "P1", "D": "P1"},  # 数据丢失证据一致
}  # 结束五起事故投票矩阵
print("事故输入：id | truth | summary")  # 展示共识系统待判断的真实事件
for incident in incidents:  # 逐条输出五起事故
    print(f"{incident['id']} | {incident['truth']} | {incident['summary']}")  # 呈现人工标签和影响范围
print("Agent 渠道：", [(agent["id"], agent["source"], agent["description"]) for agent in agents])  # 展示 A/B 的证据相关性


事故输入：id | truth | summary
I1 | P1 | 全站支付失败率 82%，持续 12 分钟
I2 | P2 | API 延迟升高但成功率正常，影响约 18% 请求
I3 | P3 | 单一企业租户无法下载账单
I4 | P4 | 计划维护窗口内监控告警，无用户影响
I5 | P1 | 数据库误删导致订单状态不可恢复
Agent 渠道： [('A', 'metrics', '主监控 Dashboard'), ('B', 'metrics', '同一 Dashboard 的摘要副本'), ('C', 'support', '客服影响工单'), ('D', 'audit', '部署与数据审计')]


## Baseline / 基线：按 Agent 身份简单多数投票

Baseline 把四票视为独立。发生 2–2 平票时按更严重等级打破平局，导致 A/B 的同源错误被计算两次。

In [2]:
from collections import Counter  # 使用计数器实现多数投票和证据簇投票
severity_order = {"P1": 1, "P2": 2, "P3": 3, "P4": 4}  # 定义平票时偏向更严重级别的排序
def majority(agent_votes):  # 按 Agent 数量计算简单多数
    counts = Counter(agent_votes.values())  # 统计每个严重级别的票数
    return sorted(counts, key=lambda level: (-counts[level], severity_order[level]))[0]  # 票数相同则选择更严重级别
baseline_rows = []  # 收集五起事故的多数投票结果
print("简单多数：incident | votes | decision | truth")  # 输出逐事件基线行为
for incident in incidents:  # 对五起事故分别投票
    decision = majority(votes[incident["id"]])  # 把四个 Agent 当作四个独立观察
    baseline_rows.append((incident["id"], decision, incident["truth"]))  # 保存决定和人工标签
    print(f"{incident['id']} | {votes[incident['id']]} | {decision} | {incident['truth']}")  # 展示同源票如何影响平局
baseline_accuracy = sum(decision == truth for _, decision, truth in baseline_rows) / len(baseline_rows)  # 计算简单多数准确率
print(f"简单多数准确率：{baseline_accuracy:.1%}")  # 输出离线同一标签下的基线指标


简单多数：incident | votes | decision | truth
I1 | {'A': 'P1', 'B': 'P1', 'C': 'P1', 'D': 'P1'} | P1 | P1
I2 | {'A': 'P1', 'B': 'P1', 'C': 'P2', 'D': 'P2'} | P1 | P2
I3 | {'A': 'P2', 'B': 'P2', 'C': 'P3', 'D': 'P3'} | P2 | P3
I4 | {'A': 'P2', 'B': 'P2', 'C': 'P4', 'D': 'P4'} | P2 | P4
I5 | {'A': 'P1', 'B': 'P1', 'C': 'P1', 'D': 'P1'} | P1 | P1
简单多数准确率：40.0%


## 核心实现：投票相关性与证据来源聚类

先计算 Agent 两两历史一致率，再按 source 聚类。A/B 的两票只形成一个 metrics 证据簇；support 和 audit 各自保留一票。

In [3]:
agent_ids = [agent["id"] for agent in agents]  # 固定相关性矩阵的 Agent 顺序
agreement = {}  # 保存每对 Agent 在五起事故上的投票一致率
for left in agent_ids:  # 枚举相关性矩阵行
    for right in agent_ids:  # 枚举相关性矩阵列
        agreement[(left, right)] = sum(votes[incident["id"]][left] == votes[incident["id"]][right] for incident in incidents) / len(incidents)  # 计算历史判断一致比例
source_by_agent = {agent["id"]: agent["source"] for agent in agents}  # 建立 Agent 到证据渠道的映射
def source_consensus(agent_votes):  # 按独立证据来源而非 Agent 身份计算共识
    source_votes = {}  # 收集每个来源簇的单一判断
    for agent_id, level in agent_votes.items():  # 遍历当前事故四个 Agent 的票
        source = source_by_agent[agent_id]  # 获取该票对应的真实证据渠道
        source_votes.setdefault(source, []).append(level)  # 把同源 Agent 放入同一簇
    collapsed = {}  # 保存每个证据簇折叠后的票
    for source, levels in source_votes.items():  # 逐来源生成一个独立判断
        collapsed[source] = majority({str(index): level for index, level in enumerate(levels)})  # 同源内部先归并为一票
    decision = majority(collapsed)  # 对三个独立证据来源执行最终多数
    return decision, collapsed  # 返回共识和可审计簇级投票
print("Agent 投票一致率矩阵：")  # 输出识别相关 Agent 的中间证据
print("    " + " ".join(f"{agent_id:>4}" for agent_id in agent_ids))  # 输出矩阵列名
for left in agent_ids:  # 逐行展示四个 Agent
    print(f"{left:>3} " + " ".join(f"{agreement[(left, right)]:4.1f}" for right in agent_ids))  # 展示 A/B 一致率为一
i2_decision, i2_sources = source_consensus(votes["I2"])  # 对共享监控过度升级的 I2 做证据聚类
print("I2 聚类投票：", i2_sources, "decision=", i2_decision)  # 展示两张 metrics 票折叠为一个来源


Agent 投票一致率矩阵：
       A    B    C    D
  A  1.0  1.0  0.4  0.4
  B  1.0  1.0  0.4  0.4
  C  0.4  0.4  1.0  1.0
  D  0.4  0.4  1.0  1.0
I2 聚类投票： {'metrics': 'P1', 'support': 'P2', 'audit': 'P2'} decision= P2


## 失败案例与修正：伪装成不同 Agent 的同一段证据

即使 Agent 声称来自不同工具，只要 evidence fingerprint 相同，仍应视为相关票。下面三个 Agent 复制同一条过期状态摘要，身份多数会形成错误 P1；按内容指纹聚类后只算一票。

In [4]:
import hashlib  # 使用内容摘要识别伪独立证据
copied_evidence = {"E1": "缓存的旧状态：支付失败率80%", "E2": "缓存的旧状态：支付失败率80%", "E3": "缓存的旧状态：支付失败率80%", "E4": "实时工单：仅延迟升高，无支付失败"}  # 构造三份复制证据和一份实时证据
copied_votes = {"E1": "P1", "E2": "P1", "E3": "P1", "E4": "P2"}  # 复制证据产生三张相同错误票
fingerprints = {agent_id: hashlib.sha256(text.encode("utf-8")).hexdigest() for agent_id, text in copied_evidence.items()}  # 为每段实际证据生成稳定指纹
naive_copied_decision = majority(copied_votes)  # 按 Agent 身份得到错误 P1 多数
clustered_by_fingerprint = {}  # 收集每个唯一内容指纹的一票
for agent_id, level in copied_votes.items():  # 遍历四个伪独立 Agent
    clustered_by_fingerprint.setdefault(fingerprints[agent_id], level)  # 完全相同证据只保留一次判断
fingerprint_counts = Counter(clustered_by_fingerprint.values())  # 统计独立证据指纹级票数
safe_copied_decision = "abstain" if len(fingerprint_counts) > 1 and len(set(fingerprint_counts.values())) == 1 else max(fingerprint_counts, key=fingerprint_counts.get)  # 独立证据平票时选择拒绝自动定级
print("复制证据指纹前缀：", {agent_id: digest[:8] for agent_id, digest in fingerprints.items()})  # 展示三份证据内容完全相同
print("按 Agent 身份：", copied_votes, "decision=", naive_copied_decision)  # 展示相关错误形成虚假多数
print("按证据指纹：", dict(fingerprint_counts), "decision=", safe_copied_decision)  # 展示独立证据不足时安全 abstain


复制证据指纹前缀： {'E1': '305889a7', 'E2': '305889a7', 'E3': '305889a7', 'E4': '15edc244'}
按 Agent 身份： {'E1': 'P1', 'E2': 'P1', 'E3': 'P1', 'E4': 'P2'} decision= P1
按证据指纹： {'P1': 1, 'P2': 1} decision= abstain


## 结果表：简单多数与来源聚类共识

In [5]:
core_rows = []  # 收集五起事故的证据来源共识结果
print("incident | truth | agent_majority | source_votes | source_consensus")  # 输出逐事件决策对照
for incident in incidents:  # 对五起事故应用同一聚类规则
    baseline_decision = majority(votes[incident["id"]])  # 获取按 Agent 身份的多数结果
    core_decision, source_votes = source_consensus(votes[incident["id"]])  # 获取按独立来源折叠的结果
    core_rows.append((incident["id"], core_decision, incident["truth"]))  # 保存核心方案和人工标签
    print(f"{incident['id']} | {incident['truth']} | {baseline_decision} | {source_votes} | {core_decision}")  # 展示 A/B 折叠后的变化
core_accuracy = sum(decision == truth for _, decision, truth in core_rows) / len(core_rows)  # 计算来源聚类共识准确率
print(f"准确率：agent_majority={baseline_accuracy:.1%}，source_consensus={core_accuracy:.1%}")  # 输出同一离线集汇总
print(f"A/B 一致率={agreement[('A', 'B')]:.1%}，但独立来源数={len(set(source_by_agent.values()))}")  # 强调 Agent 数与独立证据数不同


incident | truth | agent_majority | source_votes | source_consensus
I1 | P1 | P1 | {'metrics': 'P1', 'support': 'P1', 'audit': 'P1'} | P1
I2 | P2 | P1 | {'metrics': 'P1', 'support': 'P2', 'audit': 'P2'} | P2
I3 | P3 | P2 | {'metrics': 'P2', 'support': 'P3', 'audit': 'P3'} | P3
I4 | P4 | P2 | {'metrics': 'P2', 'support': 'P4', 'audit': 'P4'} | P4
I5 | P1 | P1 | {'metrics': 'P1', 'support': 'P1', 'audit': 'P1'} | P1
准确率：agent_majority=40.0%，source_consensus=100.0%
A/B 一致率=100.0%，但独立来源数=3


## 结果解读

A 与 B 在五起事故上 100% 一致，因为它们读取同一个 Dashboard，不应被当作两份独立证据。I2–I4 的 2–2 身份投票被严重等级规则错误打破；折叠 metrics 后，support 与 audit 的两份独立证据胜出。复制证据案例进一步说明 source 名称也可能伪造，内容指纹和平票 abstain 是必要门禁。

## 生产边界

真实共识需要校准每个来源的条件准确率、时间新鲜度、覆盖范围和共同故障模式。内容指纹只能识别完全复制，语义改写需更强 provenance。自动定级应输出置信度和反对证据，并保留值班负责人最终权责。本例使用人工构造投票，不能证明多 Agent 本身提高推理能力。

## 最小回归测试

In [6]:
assert len(incidents) >= 5 and len(agents) >= 3  # 保证共识案例包含多起事故和多个证据渠道
assert agreement[("A", "B")] == 1.0  # 保证 A/B 的完全相关性可以被明确观察
assert i2_sources == {"metrics": "P1", "support": "P2", "audit": "P2"}  # 保证同源监控票被折叠为一票
assert i2_decision == "P2"  # 保证 I2 由两份独立证据纠正过度升级
assert naive_copied_decision == "P1" and safe_copied_decision == "abstain"  # 保证复制证据虚假多数被安全拒答修正
assert core_accuracy > baseline_accuracy  # 保证来源聚类在同一五起事故上提高准确率
